# The stability of factorizations

Conditioning belongs to the *problem*; stability belongs to the *algorithm*.
The conditioning notebook showed that, for a backward-stable solve, κ·u is
the scale of the worst-case forward error. A particular solve can do better,
an exactly representable system even exactly. This one asks the other
question: does the algorithm stay within that scale, or does it lose accuracy
of its own?

The yardstick is the **backward error**. A computed solution x̂ of A x = b is
the exact solution of a nearby problem (A + ΔA) x̂ = b + Δb. The normwise
backward error

$$\eta(\hat x) = \frac{\lVert b - A\hat x\rVert}{\lVert A\rVert\,\lVert \hat x\rVert + \lVert b\rVert}$$

is the size of the smallest such perturbation, relative to the data (Rigal and
Gaches). An algorithm is **backward stable** if η is a modest multiple of u for
every input. When κη < 1, the relative forward error is then at most
2κη/(1 − κη), about 2κη when κη is small: conditioning decides how much of the
backward error can become forward error. That is a worst-case scale, not a
prediction for each solve.

This notebook:

1. **Separates the two errors:** backward error stays at u while forward error
   climbs with κ, in each of the eight formats.
2. **Measures the growth of partial pivoting**, the quantity that decides whether partial
   pivoting is stable, and shows Wilkinson's matrix, where it isn't.
3. **Reads pivot profiles**, which reflect, roughly, how a matrix's singular
   values are spread.
4. **Compares the symmetric indefinite solvers:** Cholesky refuses, unpivoted
   LDLᵀ can be unstable, and Bunch–Kaufman is stable.

Every random matrix has a seed, so the figures are the same on every build.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp

import mtl5
import mtl5.sparse as ms

g = mtl5.generators

# Chart styling, shared with the other notebooks.
SURFACE = "#fcfcfb"
INK, INK_2, RULE = "#0b0b0b", "#52514e", "#a9a8a3"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

FAMILY = {"IEEE": SERIES[0], "posit": SERIES[1], "takum": SERIES[2], "cascade": SERIES[3]}
MARKER = {"IEEE": "o", "posit": "s", "takum": "^", "cascade": "D"}
WIDTHS = {
    "16 bits": [("fp16", "IEEE"), ("posit16", "posit"), ("takum16", "takum")],
    "32 bits": [("f32", "IEEE"), ("posit32", "posit"), ("takum32", "takum")],
    "64 bits and more": [("f64", "IEEE"), ("dd_cascade", "cascade")],
}
FORMATS = [name for group in WIDTHS.values() for name, _ in group]


def as_array(x):
    # Always float64: f32's to_numpy() returns float32, and the error measures
    # below must not be evaluated in the arithmetic they are measuring.
    return np.asarray(x.to_numpy() if hasattr(x, "to_numpy") else x, dtype=np.float64)


def residual(A, x, b):
    # Each component b_i - a_i^T x, computed exactly and rounded once to float64,
    # with mtl5.mixed.dot's exact accumulator. A residual evaluated in float64
    # could round to zero below the f64 backward errors being measured.
    ones = np.array([1.0])
    return np.array(
        [
            mtl5.mixed.dot(
                mtl5.vector(np.concatenate([[b_i], -a_i])),
                mtl5.vector(np.concatenate([ones, x])),
                accumulator="exact",
            )
            for a_i, b_i in zip(A, b)
        ]
    )


def backward_error(A, x, b):
    # Normwise backward error (Rigal-Gaches), with an exactly computed residual.
    return np.linalg.norm(residual(A, x, b)) / (
        np.linalg.norm(A, 2) * np.linalg.norm(x) + np.linalg.norm(b)
    )


def unit_roundoff(dtype):
    for k in range(1, 53):
        if mtl5.convert(np.array([1.0 + 2.0**-k]), dtype).to_numpy()[0] == 1.0:
            return 2.0**-k
    return 2.0**-53


U = {name: unit_roundoff(name) for name in FORMATS}
U["dd_cascade"] = 2.0**-106  # beyond what a float64 probe can see

print(mtl5.__version__)

## 1. Backward error against forward error

For each κ from 1 to 10¹⁶, LU with partial pivoting (`mtl5.solve`) solves three
seeded `randsvd` systems in each format.

The backward error is measured against the problem the format was actually
given: A and b *as stored* in that format. That separates the algorithm from
the rounding of the input. The residual b − Ax̂ is computed exactly, so f64's
backward error is measured at its own unit roundoff.

The f64 forward error is measured against the solution of that same stored
problem computed in quad-double. The reference crosses back to Python rounded
to float64, which limits the measurement to errors above about u. That only
matters at κ = 1, where f64's error is a few u. `dd_cascade`'s forward error
is far below that, so only f64's is plotted.

In [ ]:
n = 24
kappas = np.logspace(0, 16, 9)
seeds = (1, 2, 3)

bwd = {name: np.empty((len(kappas), len(seeds))) for name in FORMATS}
fwd_f64 = np.empty((len(kappas), len(seeds)))
for i, k in enumerate(kappas):
    for j, seed in enumerate(seeds):
        A = g.randsvd(n, n, float(k), 3, seed=100 * seed + i).to_numpy()
        b = A @ np.random.default_rng(seed).standard_normal(n)
        for name in FORMATS:
            As = as_array(mtl5.convert(A, name))  # the stored problem, in float64
            bs = as_array(mtl5.convert(b, name))
            x = as_array(mtl5.solve(mtl5.convert(As, name), mtl5.convert(bs, name)))
            bwd[name][i, j] = backward_error(As, x, bs)
            if name == "f64":
                # The quad-double reference, rounded to float64 to compare with x.
                ref = as_array(
                    mtl5.solve(mtl5.convert(As, "qd_cascade"), mtl5.convert(bs, "qd_cascade"))
                )
                fwd_f64[i, j] = np.linalg.norm(x - ref) / np.linalg.norm(ref)

FLOOR = 1e-17  # errors below float64's resolution read as 0
fig, ax = plt.subplots(figsize=(7.2, 3.9), constrained_layout=True)
med_b, med_f = np.median(bwd["f64"], axis=1), np.median(fwd_f64, axis=1)
ax.loglog(
    kappas,
    np.maximum(med_f, FLOOR),
    color=SERIES[1],
    marker="o",
    markersize=5,
    linewidth=2,
    label="forward error",
    zorder=3,
)
ax.loglog(
    kappas,
    np.maximum(med_b, FLOOR),
    color=SERIES[0],
    marker="s",
    markersize=5,
    linewidth=2,
    label="backward error η",
    zorder=3,
)
ax.loglog(
    kappas,
    np.minimum(kappas * U["f64"], 10),
    color=RULE,
    linewidth=1.5,
    linestyle="--",
    label="κ·u",
    zorder=2,
)
ax.axhline(U["f64"], color=INK_2, linewidth=1, linestyle=":", zorder=2)
ax.text(kappas[-1], U["f64"] * 0.25, "u (f64)", fontsize=8, color=INK_2, ha="right", va="top")
ax.set_xlabel("κ (randsvd, mode 3)")
ax.set_ylabel("error (median of three seeds)")
ax.set_title("f64 LU: the backward error stays at u, the forward error follows κ·u", fontsize=10)
ax.legend(frameon=False, fontsize=8, loc="upper left")
plt.show()

Across sixteen decades of κ, the f64 backward error stays between about 0.2u
and 2u, while the forward error climbs from 4·10⁻¹⁶ to 3·10⁻², a decade or so
under κ·u. LU with partial pivoting is backward stable on these matrices. All
of the lost accuracy is the conditioning of the problem, none of it the
algorithm. The same holds in every format:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.7), sharey=True, constrained_layout=True)
for ax, (title, group) in zip(axes, WIDTHS.items()):
    for name, fam in group:
        if name == "dd_cascade":
            continue  # unresolved: its solution crosses to Python as float64
        ax.loglog(
            kappas,
            np.maximum(np.median(bwd[name], axis=1), FLOOR),
            color=FAMILY[fam],
            marker=MARKER[fam],
            markersize=5,
            linewidth=1.5,
            label=name,
            zorder=3,
        )
        ax.axhline(U[name], color=FAMILY[fam], linewidth=1, linestyle=":", zorder=2)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("κ (randsvd, mode 3)")
    ax.legend(frameon=False, fontsize=8, loc="best")
axes[0].set_ylabel("backward error η (median)")
axes[0].set_ylim(1e-18, 1)
fig.suptitle(
    "Backward error of LU in each format, against its unit roundoff (dotted)",
    color=INK,
    fontsize=11,
)
plt.show()

for name in FORMATS:
    if name == "dd_cascade":
        print(f"{name:11s} unresolved: its solution crosses to Python rounded to float64")
        continue
    ratio = np.median(bwd[name], axis=1).max() / U[name]
    print(
        f"{name:11s} u = {U[name]:8.1e}   largest median eta = "
        f"{np.median(bwd[name], axis=1).max():8.1e}   ({ratio:.1g} u)"
    )

On these tested matrices, every format the experiment can resolve keeps its
median backward error (over three seeds) within 2 to 4 times its own unit
roundoff, across the whole range of κ. IEEE, posit and takum arithmetic behave
alike: the number system changes u, and nothing else about the picture. A
median over a few matrices isn't a proof of backward stability for every
input; section 2 shows a matrix where partial pivoting fails.

`dd_cascade` is left out. Its solution crosses back to Python rounded to
float64, which discards the very digits its backward error lives in.

## 2. The growth factor of partial pivoting

The backward error of Gaussian elimination is bounded by a modest function of
n times ρ·u, where ρ is the **growth factor**: the largest entry produced during
elimination, at any step, relative to the largest entry of A. The factor
`mtl5.sparse.splu` exposes is the final U, so what is measured here is
ρ_U = max|U| / max|A|. That is a lower bound on ρ, since an intermediate
entry can be larger than any entry of the final U. With natural ordering and
threshold 1.0 the factorization is classic partial pivoting.

Partial pivoting keeps every entry of L at most 1 in magnitude, but ρ can
still reach 2ⁿ⁻¹. Wilkinson's matrix reaches it, and there the largest entry
sits in the final U, so ρ_U = ρ. Wilkinson's matrix reaches it: ones on the diagonal, −1
below it, and ones in the last column. Random matrices come nowhere near.

In [ ]:
def lu_factors(A):
    # Partial pivoting: natural column order, pivot threshold 1.0.
    F = ms.splu(ms.from_scipy(sp.csr_matrix(A)), ordering="natural", threshold=1.0)
    return ms.to_scipy(F.L).toarray(), ms.to_scipy(F.U).toarray()


def growth(A):
    # Final-U growth rho_U = max|U| / max|A|, a lower bound on the growth factor.
    _, Uf = lu_factors(A)
    return np.abs(Uf).max() / np.abs(A).max()


def wilkinson_growth_matrix(n):
    W = np.eye(n) - np.tril(np.ones((n, n)), -1)
    W[:, -1] = 1.0
    return W


sizes = (8, 16, 32, 64, 128, 256)
trials = 20
rho = {
    n: [growth(np.random.default_rng(1000 * n + s).standard_normal((n, n))) for s in range(trials)]
    for n in sizes
}
med = np.array([np.median(rho[n]) for n in sizes])
slope = np.polyfit(np.log(sizes), np.log(med), 1)[0]

fig, ax = plt.subplots(figsize=(7.2, 4.0), constrained_layout=True)
for n in sizes:
    ax.scatter(np.full(trials, n), rho[n], s=14, color=SERIES[0], alpha=0.45, linewidth=0, zorder=3)
ax.loglog(
    sizes,
    med,
    color=SERIES[0],
    linewidth=2,
    marker="o",
    markersize=5,
    label=f"random normal, median of {trials}",
    zorder=4,
)
wn = np.arange(4, 41, 4)
ax.loglog(
    wn,
    [growth(wilkinson_growth_matrix(k)) for k in wn],
    color=SERIES[1],
    linewidth=2,
    marker="D",
    markersize=4,
    label="Wilkinson's matrix: 2ⁿ⁻¹",
    zorder=4,
)
ax.set_xlabel("n")
ax.set_ylabel("final-U growth ρ_U = max|U| / max|A|")
ax.set_ylim(1, 1e13)
ax.set_title(
    "Partial pivoting: typical growth is small, the worst case is exponential", fontsize=10
)
ax.legend(frameon=False, fontsize=8, loc="upper left")
plt.show()

print(f"median growth: {dict(zip(sizes, np.round(med, 1)))}")
print(f"fitted slope of log(median) against log(n): {slope:.2f}")
print(f"largest growth in any random trial: {max(max(v) for v in rho.values()):.1f}")

On random matrices the final-U growth is small and slow. The median rises from 1.3 at
n = 8 to 8.2 at n = 256, a slope of 0.55 on the log–log axes, and no trial out
of 120 exceeds 12.1. ρ_U is only a lower bound on ρ, but the literature
reports similarly slow growth of ρ itself for random matrices, roughly n^{1/2}
to n^{2/3}. Wilkinson's
matrix reaches 2ⁿ⁻¹ exactly, 5.2·10⁵ already at n = 20.

What does that growth do to a solve? The matrix is perfectly well
conditioned, so the answer is pure algorithm:

In [ ]:
print(f"{'n':>4s} {'kappa(W)':>9s} {'growth':>10s} {'eta, LU':>9s} {'eta, QR':>9s}")
for n in (20, 40, 60):
    W = wilkinson_growth_matrix(n)
    b = W @ np.random.default_rng(1).standard_normal(n)
    x_lu = as_array(mtl5.solve(W, b))
    x_qr = as_array(mtl5.qr(mtl5.matrix(W)).solve(mtl5.vector(b)))
    print(
        f"{n:4d} {np.linalg.cond(W):9.1f} {growth(W):10.1e}"
        f" {backward_error(W, x_lu, b):9.1e} {backward_error(W, x_qr, b):9.1e}"
    )

κ(W) never exceeds 27, yet the backward error of LU rises from 2.6·10⁻¹³ at
n = 20 to 6.6·10⁻⁹ at n = 40 and 3.3·10⁻² at n = 60, a total loss of accuracy.
Householder QR solves the same systems with a backward error of at most
2.4·10⁻¹⁶.

Partial pivoting is backward stable *in practice*, because matrices like
Wilkinson's almost never occur, but not in the worst case. Householder QR is
backward stable unconditionally, for about twice the work. Complete or rook
pivoting bound the growth for LU at extra cost. A growth factor is cheap to
monitor, and a large one is the warning.

## 3. Pivot profiles

The pivots |uₖₖ| are a by-product of every factorization, and their profile
says something about the matrix. Their product is |det A|. The last pivot is
bounded below: |uₙₙ| ≥ σ_min(U) ≥ σ_min(A)/‖L‖₂, and partial pivoting keeps
every |lᵢⱼ| ≤ 1, so ‖L‖₂ ≤ n. What signals near-singularity is a final pivot
small *relative to the matrix*, |uₙₙ|/‖A‖₂: 10⁻²⁰·I has a tiny last pivot and
κ = 1. Every matrix below has ‖A‖₂ = σ₁ = 1, so its pivots read directly as
relative sizes. But LU with partial pivoting isn't rank revealing in
general: the pivots follow the singular values only loosely.

The chart compares the pivots with the singular values for three `randsvd`
spectra with the same κ = 10¹⁰.

In [ ]:
n = 16
mode_names = {1: "one large", 2: "one small", 3: "geometric"}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True, constrained_layout=True)
for ax, (mode, label) in zip(axes, mode_names.items()):
    A = g.randsvd(n, n, 1e10, mode, seed=4).to_numpy()
    _, Uf = lu_factors(A)
    piv = np.abs(np.diag(Uf))
    sv = mtl5.svdvals(mtl5.matrix(A))
    idx = np.arange(1, n + 1)
    ax.semilogy(
        idx,
        sv,
        color=RULE,
        linewidth=1.5,
        linestyle="--",
        marker="o",
        markersize=3,
        label="singular values σₖ",
        zorder=2,
    )
    ax.semilogy(
        idx,
        piv,
        color=SERIES[0],
        linewidth=2,
        marker="s",
        markersize=4,
        label="pivots |uₖₖ|",
        zorder=3,
    )
    ax.set_title(f"mode {mode}: {label}", fontsize=10)
    ax.set_xlabel("k")
    print(
        f"mode {mode}: last pivot {piv[-1]:.1e}, sigma_min {sv[-1]:.1e},"
        f" prod pivots / prod sigma = {np.prod(piv) / np.prod(sv):.3f}"
    )
axes[0].set_ylabel("magnitude")
axes[0].legend(frameon=False, fontsize=8, loc="upper right")
fig.suptitle(
    "Pivots of partial-pivoting LU against singular values, κ = 1e10", color=INK, fontsize=11
)
plt.show()

The product of the pivots matches the product of the singular values to
three digits in all three cases, as |det A| requires. Beyond that the
agreement is loose:

- **Modes 1 and 2:** the pivots follow the singular values closely, but the
  last pivot overstates σ_min, by 2 times in mode 1 and 40 times in mode 2.
- **Mode 3:** the pivots zigzag around the geometric decay, up to a decade
  off on either side, and the last pivot is 29 times σ_min.

A final pivot that is small relative to ‖A‖₂ is a cheap warning of
near-singularity, not a measurement of it. When the rank or σ_min itself is needed, a rank-revealing factorization
is the tool, such as QR with column pivoting or the SVD.

## 4. Symmetric indefinite systems

Three ways to factor a symmetric matrix:

- **Cholesky**, A = LLᵀ, needs positive definiteness.
- **`mtl5.ldlt`**, A = LDLᵀ without pivoting, handles an indefinite matrix as
  long as every pivot is nonzero, but it has no control over growth. Some
  nonsingular matrices have no such factorization at all: [[0, 1], [1, 0]]
  needs a zero first pivot.
- **`mtl5.bunch_kaufman`** pivots symmetrically with 1×1 and 2×2 blocks, and
  that bounds the growth.

The first experiment drifts the smallest eigenvalue τ of a seeded `randsym`
matrix through zero, so the inertia changes from (8, 0, 0) to (7, 1, 0).

In [ ]:
others = [1.0, 0.8, 0.6, 0.5, 0.4, 0.3, 0.2]
taus = np.concatenate([np.logspace(-1, -12, 12), [0.0], -np.logspace(-12, -1, 12)])
solvers = {"cholesky": mtl5.cholesky, "ldlt": mtl5.ldlt, "bunch_kaufman": mtl5.bunch_kaufman}

rows = []
for tau in taus:
    A = g.randsym(8, others + [float(tau)], seed=7).to_numpy()
    x_true = np.ones(8)
    b = A @ x_true
    row = {"tau": tau, "kappa": np.linalg.cond(A)}
    for name, factor in solvers.items():
        try:
            F = factor(mtl5.matrix(A))
        except RuntimeError:
            row[name], row[name + "_status"] = np.nan, "refused"
            continue
        x = as_array(F.solve(mtl5.vector(b)))
        row[name], row[name + "_status"] = backward_error(A, x, b), "solved"
    rows.append(row)

refused = sorted(r["tau"] for r in rows if r["cholesky_status"] == "refused")
accepted = sorted(r["tau"] for r in rows if r["cholesky_status"] == "solved")
print(
    f"Cholesky refused tau in [{min(refused):g}, {max(refused):g}] ({len(refused)} values),"
    f" accepted tau in [{min(accepted):g}, {max(accepted):g}] ({len(accepted)} values)"
)
print(f"{'tau':>9s} {'kappa':>9s} {'eta chol':>9s} {'eta ldlt':>9s} {'eta BK':>9s}")
for r in rows[::4]:
    print(
        f"{r['tau']:9.1e} {r['kappa']:9.1e} {r['cholesky']:9.1e} {r['ldlt']:9.1e}"
        f" {r['bunch_kaufman']:9.1e}"
    )

Cholesky accepts every positive τ, down to 10⁻¹², where κ = 10¹², and refuses
from τ = 0 on, including the singular semidefinite case. LDLᵀ and
Bunch–Kaufman keep backward errors around u throughout, even at τ = 0, where
κ ≈ 2·10¹⁶. Backward stability doesn't depend on the conditioning. Unpivoted
LDLᵀ is fine here only because this matrix happens to have no small leading
entries.

The second experiment gives it one. A saddle-point matrix [[δ, cᵀ], [c, M]],
with M SPD, stays well conditioned as δ → 0, but δ is unpivoted LDLᵀ's first
pivot, and the Schur complement M − ccᵀ/δ grows like 1/δ:

In [ ]:
m = 7
M = g.randspd(m, list(np.linspace(1.0, 2.0, m)), seed=3).to_numpy()
c = np.random.default_rng(9).standard_normal(m)
deltas = np.logspace(-1, -15, 15)

eta = {"ldlt": [], "bunch_kaufman": []}
conds, ldl_growth = [], []
for d in deltas:
    A = np.block([[np.array([[d]]), c[None, :]], [c[:, None], M]])
    b = A @ np.ones(m + 1)
    conds.append(np.linalg.cond(A))
    F = mtl5.ldlt(mtl5.matrix(A))
    ldl_growth.append(np.abs(np.asarray(F.diagonal())).max() / np.abs(A).max())
    for name in eta:
        x = as_array(solvers[name](mtl5.matrix(A)).solve(mtl5.vector(b)))
        eta[name].append(backward_error(A, x, b))

fig, ax = plt.subplots(figsize=(7.2, 3.9), constrained_layout=True)
ax.loglog(
    deltas,
    eta["ldlt"],
    color=SERIES[1],
    marker="o",
    markersize=5,
    linewidth=2,
    label="ldlt (no pivoting)",
    zorder=3,
)
ax.loglog(
    deltas,
    eta["bunch_kaufman"],
    color=SERIES[0],
    marker="s",
    markersize=5,
    linewidth=2,
    label="bunch_kaufman",
    zorder=3,
)
ax.axhline(2.0**-53, color=INK_2, linewidth=1, linestyle=":", zorder=2)
ax.text(deltas[-1], 2.0**-53 * 1.8, "u", fontsize=8, color=INK_2)
ax.invert_xaxis()
ax.set_xlabel("leading entry δ (smaller to the right)")
ax.set_ylabel("backward error η")
ax.set_title("A well-conditioned saddle-point matrix [[δ, cᵀ], [c, M]]", fontsize=10)
ax.legend(frameon=False, fontsize=8, loc="upper left")
plt.show()

print(f"kappa(A) over the sweep: {min(conds):.2f} to {max(conds):.2f}")
print(
    f"ldlt eta: {eta['ldlt'][0]:.1e} at delta = {deltas[0]:g},"
    f" {eta['ldlt'][-1]:.1e} at delta = {deltas[-1]:g};"
    f" bunch_kaufman eta at most {max(eta['bunch_kaufman']):.1e}"
)
print(
    f"ldlt growth max|D| / max|A|: {ldl_growth[0]:.1e} at delta = {deltas[0]:g},"
    f" {ldl_growth[-1]:.1e} at delta = {deltas[-1]:g}"
)
for d in (deltas[0], deltas[-1]):
    A = np.block([[np.array([[d]]), c[None, :]], [c[:, None], M]])
    print(
        f"bunch_kaufman ipiv at delta = {d:g}:",
        [int(p) for p in mtl5.bunch_kaufman(mtl5.matrix(A)).ipiv()],
    )

κ(A) stays at 3.3 across the whole sweep, so this is a perfectly
well-conditioned matrix. Unpivoted LDLᵀ's growth rises from 4.7 to
3.7·10¹⁴, and its backward error rises with it, from 4.5·10⁻¹⁶ to 2.2·10⁻²,
about growth × u. Bunch–Kaufman's backward error stays at u. Its pivot record,
printed above, holds no negative entries, which in LAPACK's convention means
every pivot block is 1×1. The record's entries run from 1 to n = 8, so the
binding numbers rows from 1, as LAPACK does. Its first entry, 4, says the first
step interchanges row and column 1 with 4: it moves the small δ out of the
pivot position instead of dividing by it.

## What to take away

- **Backward error is the measure of an algorithm.** A backward-stable solve
  has η ≈ u whatever κ is. The forward error is then at most about 2κη while
  κη is small, a worst-case scale, and the conditioning notebook's κ·u rule
  follows.
- **Stability carries across number systems.** On the tested matrices, LU's
  median backward error sits at 2 to 4 times its own u in every format the
  experiment resolves: the 16- and 32-bit IEEE, posit and takum formats, and
  f64. `dd_cascade`'s is below what the float64 export can show.
- **Partial pivoting is stable in practice, not in principle.** Its growth
  factor is small for random matrices and exponential for Wilkinson's. QR has
  no such worst case.
- **For symmetric indefinite systems, pivot.** Cholesky needs definiteness.
  Unpivoted LDLᵀ can lose everything on a well-conditioned matrix.
  Bunch–Kaufman doesn't.

### Where to go next

- The earlier notebooks of #166: `conditioning.ipynb`, `spectra_inertia.ipynb`
  and `nonnormality.ipynb`.
- `docs/guide/diagnostics.md`: pivot ratios, `pivot_perturb` and threshold
  pivoting in the sparse direct solvers.
- Coming in #166: Krylov convergence against the shape of the spectrum.